# Капстоун · ноутбук 1. Претрейн

> **Файлы для этого ноутбука.** Скачайте [course-materials.zip](https://vladkulikov.com/downloads/modern-llms/course-materials.zip) на сайте курса и распакуйте архив целиком, сохранив папку `capstone`. В Colab можно загрузить этот же ZIP, когда ноутбук предложит выбрать файлы.

*Курс «Modern LLMs» · капстоун · редакция 2026.10.3*

> **Что здесь происходит.** Мы проходим весь путь от сырых документов до
> обученной языковой модели — reference stack курса: конвейер очистки корпуса (M9),
> byte-level BPE с нуля (M2), шардирование потока токенов, расчёт бюджета по
> Шиншилле (M9) и сам забег с Muon и WSD-расписанием (M9).
>
> **Что нужно знать заранее.** Модули 2, 4, 5, 6, 7, 8, 9. Полезно, но не обязательно —
> модуль 1: в конце мы прочитаем собственную модель тем же паспортным скриптом.
>
> **Контракт Colab.** Обязательный путь работает на CPU и не требует сети:
> по умолчанию используется синтетический корпус, сгенерированный прямо здесь.
> Загрузка настоящих данных — отдельная ячейка с флагом, выключенным по
> умолчанию. Ни одно число из этого ноутбука не зависит от сети.

**Профили.** `SMOKE` (по умолчанию) — короткая проверка всего механизма.
`FULL` использует профиль M и 20× Chinchilla-style reference budget. Его реальное
время зависит от ускорителя и программного стека: режим `auto` выбирает BF16 на
поддерживаемых GPU, FP16 с gradient scaling на старых CUDA-ускорителях вроде T4
и FP32 на CPU.

При первом проходе выполняйте ячейки по порядку и сопоставляйте код с пояснениями. Во втором измените один параметр, сохраните результаты и объясните, какое утверждение опыт позволяет проверить.

## 1. Окружение

Пакет `minillm` — это код капстоуна: `config`, `model`, `tokenizer`, `corpus`,
`data`, `train`, `generate`. Ноутбук ничего не переопределяет, он только
собирает конвейер из готовых частей и показывает числа.

In [ ]:
import importlib
import sys
import subprocess
import zipfile
from pathlib import Path

ARCHIVE_GLOB = "modern-llms-capstone-*.zip"

def _activate_capstone_root(candidate: Path):
    package_init = candidate / "minillm" / "__init__.py"
    if not package_init.exists():
        return None
    root = str(candidate.resolve())
    if root not in sys.path:
        sys.path.insert(0, root)
    return root

def _find_capstone_root(search_root: Path):
    direct = _activate_capstone_root(search_root)
    if direct:
        return direct
    for package_root in (search_root / "capstone" / "minillm" / "ru", search_root / "minillm" / "ru"):
        found = _activate_capstone_root(package_root)
        if found:
            return found
    for child in sorted(search_root.glob("modern-llms-capstone-*")):
        found = _activate_capstone_root(child)
        if found:
            return found
    return None

def _safe_extract(archive_path: Path, destination: Path):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise RuntimeError(f"unsafe path in archive: {member.filename}")
        archive.extractall(destination)

def ensure_minillm():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        found = _find_capstone_root(candidate)
        if found:
            return "пакет проекта: " + found

    try:
        import minillm  # noqa: F401
        return "minillm уже доступен"
    except ImportError:
        pass

    archives = sorted(Path.cwd().glob(ARCHIVE_GLOB)) + sorted(Path.cwd().glob("course-materials.zip"))
    if not archives:
        try:
            from google.colab import files
        except ImportError:
            files = None
        if files is not None:
            print("Загрузите course-materials.zip со страницы ноутбуков курса. Структура каталогов будет восстановлена автоматически.")
            files.upload()
            archives = sorted(Path.cwd().glob(ARCHIVE_GLOB)) + sorted(Path.cwd().glob("course-materials.zip"))

    if archives:
        archive_path = archives[-1]
        destination = Path.cwd() / ".capstone_runtime"
        destination.mkdir(exist_ok=True)
        _safe_extract(archive_path, destination)
        root = _find_capstone_root(destination)
        if root:
            return "архив проекта: " + str(archive_path)
        matches = sorted(destination.glob("*/minillm/__init__.py"))
        if len(matches) != 1:
            raise RuntimeError(
                f"expected one minillm package in {archive_path.name}, "
                f"found {len(matches)}"
            )
        root = _activate_capstone_root(matches[0].parent.parent)
        if root:
            return "архив проекта: " + str(archive_path)

    raise RuntimeError("minillm не найден. Скачайте course-materials.zip со страницы ноутбуков курса и либо распакуйте его целиком, либо загрузите ZIP в текущую сессию Colab.")

print(ensure_minillm())

for package in ("torch", "numpy", "matplotlib", "regex"):
    try:
        importlib.import_module(package)
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", package],
            check=True,
        )

import os, torch, numpy as np, hashlib
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# ---------------------------------------------------------------- профиль забега
PROFILE = "SMOKE"          # "SMOKE" | "FULL"
USE_REAL_CORPUS = False    # сетевая загрузка настоящих данных — см. §3
SEED = 0

torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
from minillm.precision import resolve_compute_dtype
COMPUTE_DTYPE = resolve_compute_dtype(DEVICE, "auto")

SETTINGS = {
    "SMOKE": dict(docs=8_000, vocab_size=2_048, seq_len=128,
                  model_profile="S", steps=250, micro_batch_size=8, grad_accum=1),
    "FULL":  dict(docs=200_000, vocab_size=16_384, seq_len=512,
                  model_profile="M", steps=18_886, micro_batch_size=16, grad_accum=4),
}[PROFILE]
if os.getenv("CAPSTONE_FAST_VALIDATION") == "1":
    SETTINGS = {**SETTINGS, "docs": min(SETTINGS["docs"], 800),
                "vocab_size": min(SETTINGS["vocab_size"], 512),
                "steps": min(SETTINGS["steps"], 12),
                "micro_batch_size": min(SETTINGS["micro_batch_size"], 2),
                "grad_accum": 1}
print(PROFILE, "на", DEVICE, "с вычислениями", COMPUTE_DTYPE, "->", SETTINGS)

## 2. Корпус: воронка очистки

Порядок стадий в конвейере не произволен — от дешёвых к дорогим (M9).
Эвристики стоят проход по строке, дедуп — хеширование, модельный классификатор
качества — инференс. Поэтому классификатор видит только то, что пережило
первые ступени, и стоимость конвейера определяется ими.

Синтетический корпус ниже собран из шаблонов на двух языках и намеренно
испорчен: в нём есть дубликаты, почти-дубликаты, слишком короткие документы и
символьный мусор. Это и есть материал для воронки — на чистых данных
конвейер нечему научить.

In [ ]:
import random
from minillm.corpus import FilterConfig, pipeline

RU_SLOTS = dict(
    subject=["языковая модель", "трансформер", "механизм внимания", "токенизатор",
             "оптимизатор", "нормализация", "остаточная связь", "роутер экспертов",
             "кэш ключей", "эмбеддинг", "спекулятивный декод", "квантизация"],
    verb=["предсказывает", "обрабатывает", "сокращает", "ускоряет", "стабилизирует",
          "ограничивает", "восстанавливает", "перераспределяет", "измеряет"],
    obj=["следующий токен", "длинный контекст", "стоимость шага", "объём памяти",
         "масштаб активаций", "разброс градиентов", "поток данных", "качество ответа",
         "нагрузку экспертов", "пропускную способность"],
    tail=["на длинных последовательностях", "при обучении в низкой точности",
          "без потери качества", "за счёт дополнительной матрицы",
          "ценой роста числа параметров", "в фазе декодирования",
          "по сравнению с базовой схемой", "при неизменном бюджете вычислений"],
)
EN_SLOTS = dict(
    subject=["a language model", "the transformer", "the attention mechanism",
             "the tokenizer", "the optimizer", "normalization", "the residual stream",
             "the expert router", "the key cache", "the embedding table",
             "speculative decoding", "quantization"],
    verb=["predicts", "processes", "reduces", "accelerates", "stabilizes", "bounds",
          "restores", "redistributes", "measures"],
    obj=["the next token", "a long context", "the cost per step", "the memory footprint",
         "the scale of activations", "the spread of gradients", "the data stream",
         "the quality of the answer", "the load across experts", "the throughput"],
    tail=["on long sequences", "when training in low precision", "without losing quality",
          "at the price of one more matrix", "at the cost of more parameters",
          "during the decode phase", "compared with the baseline scheme",
          "under an unchanged compute budget"],
)


def sentence(rng: random.Random, slots: dict) -> str:
    parts = [rng.choice(slots["subject"]), rng.choice(slots["verb"]),
             rng.choice(slots["obj"]), rng.choice(slots["tail"])]
    return " ".join(parts).capitalize() + "."


def synthetic_corpus(n_docs: int, seed: int = 0) -> list[str]:
    """Двуязычный корпус со слотовой грамматикой и намеренно внесённым мусором.

    Никакой это не претрейн-корпус: грамматика одна, словарь узкий. Задача —
    дать конвейеру и токенизатору воспроизводимый, заведомо лицензионно чистый
    материал, на котором виден каждый механизм. Слоты дают комбинаторное
    разнообразие, иначе near-dup дедуп справедливо схлопнул бы весь корпус.
    """
    rng = random.Random(seed)
    docs = []
    for i in range(n_docs):
        slots = RU_SLOTS if i % 2 else EN_SLOTS
        docs.append(" ".join(sentence(rng, slots) for _ in range(rng.randint(3, 6))))
    # Мусор для воронки: точные копии, почти-копии, короткие и символьные.
    docs += docs[:n_docs // 20]                                   # точные дубли
    docs += [d + " Дополнение." for d in docs[:n_docs // 40]]     # почти-дубли
    docs += ["ок"] * (n_docs // 40)                               # короткие
    docs += ["!@#$%^&*()" * 8] * (n_docs // 40)                   # символьный мусор
    rng.shuffle(docs)
    return docs

raw_docs = synthetic_corpus(SETTINGS["docs"], seed=SEED)
print(f"сырых документов: {len(raw_docs):,}")

In [ ]:
clean_docs, report = pipeline(raw_docs, FilterConfig(min_chars=60),
                              near_dup_threshold=0.85)
print(report)
print(f"\nитог: {len(clean_docs):,} документов, "
      f"{sum(len(d) for d in clean_docs):,} символов")

# Разделяем ДО обучения tokenizer: validation не участвует ни в словаре,
# ни в merge table. Глобальная очистка выполнена до split, поэтому точные и
# near-дубли не расходятся между train и validation.
split_rng = random.Random(SEED + 17)
split_rng.shuffle(clean_docs)
valid_count = max(1, round(0.05 * len(clean_docs)))
valid_docs = clean_docs[:valid_count]
train_docs = clean_docs[valid_count:]
print(f"document split: train={len(train_docs):,}, valid={len(valid_docs):,}")

Воронка показывает, какие документы остались и почему были отброшены остальные. Большая доля отказов сама по себе не доказывает ни качество фильтра, ни ошибку порога. Возьмите примеры по обе стороны границы и отдельно сравните языки и типы документов: один порог может хорошо удалять мусор и одновременно терять полезные короткие тексты. В нашем корпусе причины внесены намеренно, поэтому их можно сопоставить с отчётом.

## 3. Настоящие данные (опционально)

Обязательный путь ноутбука настоящих данных **не требует** и ни одного
опубликованного числа из них не берёт. Ячейка ниже включается флагом
`USE_REAL_CORPUS` из §1.

Проверенные варианты с допустимыми лицензиями:

| Корпус | Лицензия | Заметка |
|---|---|---|
| `HuggingFaceFW/fineweb-2`, подмножество `rus_Cyrl` | ODC-By 1.0 | атрибуция обязательна; действуют также условия использования CommonCrawl |
| `roneneldan/TinyStories` | CDLA-Sharing-1.0 | синтетические рассказы с узким словарём; share-alike распространяется на производные **датасеты**, не на обученную модель |

Размер модели сам по себе не определяет связность текста. TinyStories задаёт более узкую задачу, чем широкий веб-корпус; результат зависит также от бюджета обучения, покрытия данных и способа оценки. Сравнивайте отложенный loss и реальные продолжения по зафиксированному протоколу. Внешний корпус остаётся необязательным расширением.

In [ ]:
if USE_REAL_CORPUS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)
    from datasets import load_dataset

    DATA_REVISIONS = {
        "fineweb2": "v2.1.1",
        "tinystories": "f54c09fd23315a6f9c86f9dc80f725de7d8f9c64",
    }
    train_target = SETTINGS["docs"] // 2
    eval_target = max(100, SETTINGS["docs"] // 40)

    ru_train = load_dataset("HuggingFaceFW/fineweb-2", name="rus_Cyrl",
                            split="train", streaming=True,
                            revision=DATA_REVISIONS["fineweb2"])
    ru_eval = load_dataset("HuggingFaceFW/fineweb-2", name="rus_Cyrl",
                           split="test", streaming=True,
                           revision=DATA_REVISIONS["fineweb2"])
    en_train = load_dataset("roneneldan/TinyStories", split="train", streaming=True,
                            revision=DATA_REVISIONS["tinystories"])
    en_eval = load_dataset("roneneldan/TinyStories", split="validation", streaming=True,
                           revision=DATA_REVISIONS["tinystories"])

    def take_text(dataset, limit):
        rows = []
        for row in dataset:
            rows.append(row["text"])
            if len(rows) >= limit:
                break
        return rows

    raw_train = take_text(ru_train, train_target) + take_text(en_train, train_target)
    raw_valid = take_text(ru_eval, eval_target) + take_text(en_eval, eval_target)
    train_docs, train_report = pipeline(raw_train, FilterConfig(), near_dup_threshold=0.85)
    valid_docs, valid_report = pipeline(raw_valid, FilterConfig(), near_dup_threshold=0.85)
    # Защита от точной утечки между источниками/splits после нормализации.
    train_fingerprints = {hashlib.sha256(d.encode("utf-8")).digest() for d in train_docs}
    valid_docs = [d for d in valid_docs
                  if hashlib.sha256(d.encode("utf-8")).digest() not in train_fingerprints]
    print("train:", train_report)
    print("valid:", valid_report)
    print("revisions:", DATA_REVISIONS)
    print("\nАтрибуция: FineWeb2 (ODC-By 1.0 + Common Crawl Terms), "
          "TinyStories (CDLA-Sharing-1.0). Это техническое описание, не юридическая консультация.")
else:
    DATA_REVISIONS = {"synthetic": f"generator-v1-seed-{SEED}"}
    print("USE_REAL_CORPUS = False — используем document-level split синтетического корпуса")

## 4. Токенизатор: byte-level BPE с нуля

Базовый алфавит содержит 256 байтов. Поэтому любой корректно закодированный текст UTF-8 можно представить без неизвестного токена, но начальная длина зависит от байтового представления. Буквы обычного русского алфавита занимают по два байта; это не правило для всех символов кириллических блоков Unicode. Слияния BPE могут объединять байты внутри символа и между символами.

Количество и разнообразие русского текста влияют на выученные слияния вместе с размером словаря и правилами подготовки данных. Мы измерим число токенов на символ и байтов на токен на отдельных RU/EN-примерах. Эти показатели характеризуют данный словарь и данные; сами по себе они не измеряют качество будущей языковой модели.

In [ ]:
import time
from minillm.tokenizer import BBPETokenizer
from minillm.data import ByteTokenizer
from minillm.chat import CHAT_SPECIALS

# Управляющие токены чата резервируются в словаре СРАЗУ, хотя претрейну они не
# нужны. Причина ровно та же, по которой в o200k_harmony лежат 1 075
# зарезервированных идентификаторов (M2): добавить их потом — значит менять
# vocab_size и пересобирать эмбеддинги. Ноутбук 2 берёт этот же чекпойнт и
# рассчитывает найти в нём `<|eot_id|>`.
train_text = "\n".join(train_docs)
t0 = time.time()
tokenizer = BBPETokenizer.train(train_text, vocab_size=SETTINGS["vocab_size"],
                                specials=["<|bos|>", "<|eos|>"] + CHAT_SPECIALS)
print(f"обучено за {time.time() - t0:.1f} с: словарь {tokenizer.vocab_size}, "
      f"merge'ей {len(tokenizer.merges)}")
if tokenizer.vocab_size < SETTINGS["vocab_size"]:
    print('Доступные пары для слияний закончились до заданного размера словаря.')

sample = "Языковая модель предсказывает следующий токен."
assert tokenizer.decode(tokenizer.encode(sample)) == sample, "round-trip сломан"
print("round-trip:", repr(tokenizer.decode(tokenizer.encode(sample))))

In [ ]:
probe_rng = random.Random(123)
ru_probe = " ".join(sentence(probe_rng, RU_SLOTS) for _ in range(40))
en_probe = " ".join(sentence(probe_rng, EN_SLOTS) for _ in range(40))
plain = ByteTokenizer()

rows = []
for name, probe in (("RU", ru_probe), ("EN", en_probe)):
    trained = tokenizer.fertility(probe)
    baseline = len(plain.encode(probe)) / len(probe)
    rows.append((name, trained["tokens_per_char"], baseline,
                 trained["bytes_per_token"]))

print(f"{'язык':<5} {'BBPE tok/char':>14} {'byte-level tok/char':>21} {'bytes/token':>12}")
for name, trained, baseline, bpt in rows:
    print(f"{name:<5} {trained:>14.3f} {baseline:>21.3f} {bpt:>12.2f}")
print("\nbyte-level для кириллицы даёт ~2 токена на символ — это и есть цена "
      "отсутствия merge'ей для языка (M2).")

## 5. Шардирование потока

Разобьём поток токенов на файлы `.bin` — шарды — и сохраним контрольные суммы в `index.json`. Необходимая память определяется объёмом данных, а не только размером модели. Здесь данные невелики, зато можно проверить тот же порядок чтения и контроль целостности, который потребуется при большем корпусе. Граница шарда не должна терять переход к следующему токену.

In [ ]:
from minillm.data import ShardStream, TokenStream, write_shards

EOS = tokenizer.specials["<|eos|>"]

def encode_documents(documents):
    stream = []
    for doc in documents:
        stream.extend(tokenizer.encode(doc))
        stream.append(EOS)
    return stream

train_token_ids = encode_documents(train_docs)
valid_token_ids = encode_documents(valid_docs)
print(f"train tokens: {len(train_token_ids):,}; valid tokens: {len(valid_token_ids):,}")

shard_dir = Path("data_shards")
index = write_shards(train_token_ids, shard_dir / "train", shard_tokens=250_000)
write_shards(valid_token_ids, shard_dir / "valid", shard_tokens=250_000)

train_stream = ShardStream(shard_dir / "train", seq_len=SETTINGS["seq_len"], verify=True)
valid_stream = ShardStream(shard_dir / "valid", seq_len=SETTINGS["seq_len"], verify=True)
print(f"train: {len(index['shards'])} шардов, {index['total_tokens']:,} токенов, "
      f"{len(train_stream):,} окон, checksum {train_stream.checksum()}")
print(f"valid: {len(valid_stream):,} окон — документно отделён до tokenizer training")

## 6. Конфиг модели и бюджет

Прежде чем запускать забег, посчитаем три вещи, которые курс учит считать до
запуска, а не после OOM:

1. **параметры** — по формуле M6 и сверка с фактическим `model.parameters()`;
2. **KV-кэш на токен** — `2·L·H_kv·d_head·sizeof(dtype)` (M5);
3. **20× Chinchilla-style reference budget** — 20 токенов на параметр (M9), и сколько это шагов.

In [ ]:
from minillm.config import MiniLLMConfig, kv_bytes_per_token
from minillm.model import MiniLLM, MoELayer
from minillm.train import TrainConfig, memory_report

cfg = MiniLLMConfig.profile(
    SETTINGS["model_profile"],
    vocab_size=tokenizer.vocab_size,
    max_position_embeddings=max(SETTINGS["seq_len"], 256),
)
model = MiniLLM(cfg).to(DEVICE)

analytic = cfg.parameter_report()
assert analytic["total"] == model.num_parameters(), "формула и код разошлись"
print(f"параметров: {analytic['total']:,} (формула == факт)")
for key in ("embeddings", "attention", "ffn", "norms"):
    print(f"  {key:<12} {analytic[key]:>12,}  ({analytic[key] / analytic['total']:.1%})")
print(f"  блок стоит {analytic['block_over_d2']}·d²")
reference_cache_bpt = kv_bytes_per_token(cfg, dtype_bytes=2)
runtime_cache_bpt = kv_bytes_per_token(cfg, dtype_bytes={"fp32": 4, "bf16": 2, "fp16": 2}[COMPUTE_DTYPE])
print(f"\nKV-кэш, BF16/FP16 reference: {reference_cache_bpt} байт/токен "
      f"→ {reference_cache_bpt * SETTINGS['seq_len'] / 1024:.1f} KiB на окно")
print(f"KV-кэш при текущем compute dtype ({COMPUTE_DTYPE}): {runtime_cache_bpt} байт/токен")
print("постоянная память обучения:", memory_report(cfg, "adamw"))

In [ ]:
tcfg = TrainConfig(
    steps=SETTINGS["steps"],
    seq_len=SETTINGS["seq_len"],
    micro_batch_size=SETTINGS["micro_batch_size"],
    grad_accum=SETTINGS["grad_accum"],
    optimizer="muon",
    compute_dtype="auto",
    gradient_checkpointing=DEVICE.startswith("cuda"),
    seed=SEED,
    device=DEVICE,
    log_every=max(1, SETTINGS["steps"] // 10),
    out_dir="runs/pretrain",
)

chinchilla_tokens = 20 * analytic["total"]
planned_tokens = tcfg.steps * tcfg.tokens_per_step
print(f"плановых next-token целей за шаг: {tcfg.tokens_per_step:,}")
print(f"запланировано:  {planned_tokens:,} обучаемых целей за {tcfg.steps:,} шагов")
print(f"reference 20×: {chinchilla_tokens:,} токенов "
      f"→ {chinchilla_tokens / tcfg.tokens_per_step:,.0f} шагов")
print(f"доля бюджета:   {planned_tokens / chinchilla_tokens:.1%}")
print(f"эпох по корпусу: {planned_tokens / len(train_token_ids):.2f}")

Плановый бюджет считает next-token цели: у каждого окна длиной `seq_len+1` их `seq_len`. Журнал отдельно измеряет фактические входные позиции (`tokens`, включая первую позицию окна и PAD) и обучаемые цели после сдвига (`supervised_tokens`). Скорость `tokens_per_s` относится к входным позициям. Loss обучения и проверки — среднее по всем обучаемым целям выбранных батчей; при накоплении градиентов знаменатель общий для эффективного батча.

Строка «доля бюджета» сравнивает запланированные токены с ориентиром $20P$. Это полезная шкала размера опыта, но не тест достаточности обучения: оптимальный бюджет зависит от данных и цели. В `SMOKE` короткий запуск проверяет связность конвейера; `FULL` задаёт более длинный учебный план.

Число проходов по корпусу показывает степень повторного использования данных. Повторение небольшого набора повышает риск запоминания, но универсального порога в четыре эпохи нет. Сопоставляйте кривые обучения и отложенной проверки и помните, что синтетические шаблоны ограничивают разнообразие обеих частей.

## 7. Забег

Оптимизатор — гибрид: Muon на двумерных матрицах, AdamW на эмбеддингах, нормах
и роутере (M9). Расписание — WSD: разогрев, полка, спад (M9). Полка
позволяет прервать забег в любой момент и добить спад, не переобучая расписание
под новую длину.

In [ ]:
from minillm.train import train, save_checkpoint

state = train(model, train_stream, tcfg, eval_stream=valid_stream)
print(f"\nготово: {state.step} шагов, "
      f"{state.tokens:,} входных позиций, {state.supervised_tokens:,} целей, "
      f"финальный лосс {state.history[-1]['loss']:.4f}")

In [ ]:
import matplotlib.pyplot as plt

steps = [h["step"] for h in state.history]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(steps, [h["loss"] for h in state.history], label="train", color="#5E81AC")
if "eval_loss" in state.history[0]:
    axes[0].plot(steps, [h["eval_loss"] for h in state.history], label="valid",
                 color="#BF616A")
axes[0].axhline(np.log(cfg.vocab_size), ls="--", lw=1, color="#4C566A",
                label="ln(V) — равномерная модель")
axes[0].set_xlabel("шаг"); axes[0].set_ylabel("кросс-энтропия"); axes[0].legend()
axes[0].set_title("лосс")

axes[1].plot(steps, [h["lr_scale"] for h in state.history], color="#A3BE8C")
axes[1].set_xlabel("шаг"); axes[1].set_ylabel("множитель lr")
axes[1].set_title("WSD-расписание")

axes[2].plot(steps, [h["tokens_per_s"] for h in state.history], color="#D08770")
axes[2].set_xlabel("шаг"); axes[2].set_ylabel("входных позиций/с")
axes[2].set_title("пропускная способность")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

bpc = state.history[-1]["loss"] / np.log(2)
print(f"финальная кросс-энтропия {state.history[-1]['loss']:.4f} нат "
      f"= {bpc:.4f} бит на токен")
print('Кривая относится к данному бюджету и корпусу; её плато само по себе не определяет причину ограничения.')

### 7.1. Разрежённый близнец: общие и активные параметры

Теперь заменим плотный FFN банком экспертов, сохранив словарь, глубину, внимание и позиции. Ширина эксперта выбрана так, чтобы число активных параметров было близко к плотной модели. Сравним точные числа: например, у стандартных S и S-MoE это 6 327 552 и 6 593 792. В текущем запуске словарь может быть другим, и таблица пересчитает паспорт.

Близкие активные бюджеты не означают одинакового времени на токен. Маршрутизация, размеры матриц и объединение результатов тоже требуют работы. Геометрия KV-кэша при неизменном внимании совпадает точно; сначала проверим именно эти расчётные свойства.

In [ ]:
dense_cfg = cfg
moe_cfg = MiniLLMConfig(**{**MiniLLMConfig.twin(SETTINGS["model_profile"]).to_dict(),
                           "vocab_size": tokenizer.vocab_size,
                           "max_position_embeddings": cfg.max_position_embeddings})
moe_model = MiniLLM(moe_cfg).to(DEVICE)

rows = [("dense", dense_cfg, model), ("MoE", moe_cfg, moe_model)]
print(f"{'вариант':<8} {'total':>14} {'active':>12} {'разреж.':>8} "
      f"{'KV Б/ток':>9} {'FLOP/ток':>12}")
for name, c, m in rows:
    assert c.parameter_report()["total"] == m.num_parameters(), "формула разошлась с моделью"
    total, active = c.parameter_report()["total"], c.active_parameters()
    print(f"{name:<8} {total:>14,} {active:>12,} {total / active:>7.2f}× "
          f"{kv_bytes_per_token(c):>9,} {c.flops_per_token():>12,}")

print(f"\nэкспертов {moe_cfg.num_local_experts}, top-{moe_cfg.num_experts_per_tok}, "
      f"shared {moe_cfg.num_shared_experts}, первый слой плотный: "
      f"{not moe_cfg.moe_layer(0)}")
print(f"активных у MoE к плотному: "
      f"×{moe_cfg.active_parameters() / dense_cfg.active_parameters():.3f}")
print(f"KV-кэш совпадает до байта: "
      f"{kv_bytes_per_token(dense_cfg) == kv_bytes_per_token(moe_cfg)} — "
      f"внимание не трогали, сравнение чистое")

Таблица отделяет ёмкость банка от работы выбранных экспертов. Теперь обучим обе модели на одинаковом числе шагов и токенов. Сопоставим loss, скорость и распределение назначений, не предполагая заранее ни выигрыша MoE, ни равной аппаратной стоимости.

In [ ]:
moe_state = train(moe_model, train_stream, tcfg, eval_stream=valid_stream)

dense_last, moe_last = state.history[-1], moe_state.history[-1]
print(f"\n{'вариант':<8} {'train':>9} {'valid':>9} {'позиций/с':>9}")
for name, h in (("dense", dense_last), ("MoE", moe_last)):
    print(f"{name:<8} {h['loss']:>9.4f} {h.get('eval_loss', float('nan')):>9.4f} "
          f"{h['tokens_per_s']:>9.0f}")

moe_layer = next(l.ffn for l in moe_model.layers if isinstance(l.ffn, MoELayer))
load = moe_layer.expert_load
print(f"\nнагрузка экспертов последнего замера: min {load.min():.3f}, "
      f"max {load.max():.3f}, std {load.std():.3f}")
print(f"идеально равномерная была бы "
      f"{moe_cfg.num_experts_per_tok / moe_cfg.num_local_experts:.3f} на эксперта")
print(f"смещений aux-loss-free: max|b| = {moe_layer.expert_bias.abs().max():.4f}")

**Как читать сравнение.** При равномерной маршрутизации каждый из $E$ экспертов получает в среднем долю $k/E$ токенов. Реальная нагрузка может быть неравномерной; меньше токенов не означает автоматически меньше шагов оптимизатора. Таблица нагрузки помогает понять, сколько данных фактически использовалось каждым экспертом.

Один короткий запуск показывает поведение двух заданных конфигураций. Причину разницы в loss нельзя вывести только из числа параметров. В ноутбуке 4 вернёмся к этому вопросу с отдельным изменением скорости балансировки и измерением отброшенных назначений.

## 8. Что выучила модель

Сгенерируем продолжения с жадным выбором и с выборкой. Prefill обрабатывает позиции промпта, decode продолжает последовательность с KV-кэшем. М10 объясняет, при каких условиях первая фаза ограничена вычислениями, а вторая — передачей данных; это не безусловные свойства малого CPU-запуска. Здесь сохраняем фактические времена и читаем продолжения, отделяя работоспособность механизма от качества языка.

In [ ]:
from minillm.generate import SamplingConfig, generate

for temperature, top_p in ((0.0, 1.0), (0.8, 0.9)):
    prompt = tokenizer.encode("Языковая модель")
    out, stats = generate(model, prompt, 80,
                          SamplingConfig(temperature=temperature, top_p=top_p, seed=SEED),
                          stop_tokens={EOS})
    label = "жадно" if temperature == 0 else f"T={temperature}, top_p={top_p}"
    print(f"--- {label}\n{tokenizer.decode(out)!r}\n")
print("статистика последнего запуска:", stats.as_dict())

## 9. Паспорт собственной модели

Курс начался с того, что мы учились читать `config.json` чужой модели. Теперь
модель наша — и тот же паспортный подход читает её без единой правки. Это и есть
замыкание кольца: арифметика модулей 1, 5 и 6 применяется к файлу, который мы
только что написали сами.

In [ ]:
run_dir = Path("runs/pretrain/final")
save_checkpoint(run_dir, model, state.optimizer, tcfg, state)
tokenizer.save(run_dir / "tokenizer.json")
print("сохранено:", sorted(p.name for p in run_dir.iterdir()))

In [ ]:
import json

def read_passport(path: Path) -> None:
    """Паспорт модели по config.json — тот же приём, что в модуле 1."""
    c = json.loads((path / "config.json").read_text(encoding="utf-8"))
    heads, kv = c["num_attention_heads"], c["num_key_value_heads"]
    attention = "MHA" if heads == kv else ("MQA" if kv == 1 else f"GQA {heads}/{kv}")
    ffn_ratio = c["intermediate_size"] / c["hidden_size"]
    cache_bytes = {"fp32": 4, "bf16": 2, "fp16": 2}[stats.cache_dtype]
    kv_bytes = 2 * c["num_hidden_layers"] * kv * c["head_dim"] * cache_bytes

    print("Ось I  · размер     :", f"d={c['hidden_size']}, L={c['num_hidden_layers']}, "
          f"V={c['vocab_size']}, эмбеддинги "
          f"{'связаны' if c['tie_word_embeddings'] else 'раздельны'}")
    print("Ось II · контекст    :", f"{c['max_position_embeddings']} токенов, "
          f"RoPE base {c['rope_theta']:g}, растяжка "
          f"{c['rope_scaling'] or 'нет'}")
    print("Ось III· внимание    :", attention,
          f"· head_dim {c['head_dim']} · sinks {'да' if c['attention_sinks'] else 'нет'}"
          f" · окно {c['sliding_window'] or 'нет'}")
    print("Ось IV · блок        :", f"норма {c['norm_placement']}, {c['hidden_act']}, "
          f"d_ffn/d = {ffn_ratio:.2f}"
          + (f", кламп {c['swiglu_limit']}" if c["swiglu_limit"] else ""))
    print("Ось V  · разреженность:", "dense" if not c["num_local_experts"]
          else f"MoE {c['num_local_experts']} экспертов, top-{c['num_experts_per_tok']}")
    print("Производные          :", f"KV-кэш {kv_bytes} Б/токен в измеренном {stats.cache_dtype}, "
          f"{kv_bytes * c['max_position_embeddings'] / 1024 ** 2:.1f} МиБ на полный контекст")

read_passport(run_dir)

## 10. Итог и что дальше

Мы прошли путь, который курс разбирал по частям: сырые
документы → воронка очистки → собственный BPE → шарды → учебная reference-модель курса →
забег с Muon и WSD → генерация → паспорт. Все числа посчитаны здесь, ни одно не
взято из лекции на веру.

Долю выбранного бюджета относительно ориентира $20P$ мы вычислили в паспорте.
Она зависит от профиля и числа шагов, а сам ориентир не оценивает качество модели.
Синтетический корпус позволяет проследить обучение и проверить связи между
этапами. Для сравнения архитектур нужен контролируемый опыт из ноутбука 4;
для вывода о естественном языке — соответствующий корпус и оценка ответов.

**Дальше:**

- ноутбук 2 — посттрейн: chat-шаблон, маскирование лосса по assistant-спанам,
  packing и DPO;
- ноутбук 3 — инференс: префиксный кэш, спекулятивное декодирование с небольшой
  моделью-черновиком, квантизация;
- ноутбук 4 — абляции: проверка утверждений курса числом и честный раздел о
  границах масштаба.

**Атрибуция данных.** Синтетический корпус §2 сгенерирован кодом этого ноутбука.
При включённом `USE_REAL_CORPUS` действуют лицензии соответствующих датасетов:
FineWeb-2 — ODC-By 1.0 плюс условия использования CommonCrawl, TinyStories —
CDLA-Sharing-1.0.